In [ ]:
import pandas as pd
import numpy as np

# Load database CRM awal
file_path = '/content/CRM Data Dataset.xlsx'
df = pd.read_excel(file_path)
df_clean = df.copy()

# 1. STANDARISASI DAN BERSIHIN FORMAT DATA
df_clean['Last_Purchase_Date'] = pd.to_datetime(df_clean['Last_Purchase_Date'], unit='D', origin='1899-12-30').dt.strftime('%Y-%m-%d')
df_clean.loc[df_clean['Days_Since_Last_Purchase'] == 999, 'Last_Purchase_Date'] = '-'

# tracking aktivitas customer
check_cols = ['Add_To_Cart', 'WhatsApp_Clicked', 'Chat_Replied', 'Email_Opened', 'Social_Liked']
for col in check_cols:
    df_clean[col + '_flag'] = df_clean[col].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

# Ambil nilai acuan statistik berdasarkan sebaran data riil
top_view = df_clean['View_Count'].quantile(0.75)
med_freq = df_clean['Purchase_Frequency_6M'].median()
top_value = df_clean['Total_Transaction_Value'].quantile(0.75)
top_pages = df_clean['Pages_Visited'].quantile(0.75)

# 2. PROSES SKORING LEADS (LEAD SCORE & LEAD STATUS)
df_clean['Lead_Score'] = (
    (df_clean['Add_To_Cart_flag'] * 25) +
    (df_clean['WhatsApp_Clicked_flag'] * 20) +
    (df_clean['Chat_Replied_flag'] * 20) +
    (df_clean['Email_Opened_flag'] * 10) +
    (df_clean['View_Count'].apply(lambda x: 10 if x >= top_view else 0)) +
    (df_clean['Purchase_Frequency_6M'].apply(lambda x: 15 if x > med_freq else 0)) +
    (df_clean['Total_Transaction_Value'].apply(lambda x: 15 if x >= top_value else 0)) +
    (df_clean['Days_Since_Last_Purchase'].apply(lambda x: -20 if (x > 90 and x != 999) else 0))
)
df_clean['Lead_Score'] = df_clean['Lead_Score'].clip(lower=0, upper=100)

cond_status = [
    (df_clean['Lead_Score'] >= 80),
    (df_clean['Lead_Score'] >= 60) & (df_clean['Lead_Score'] < 80),
    (df_clean['Lead_Score'] >= 40) & (df_clean['Lead_Score'] < 60),
    (df_clean['Lead_Score'] < 40)
]
label_status = ['Hot Lead', 'Warm Lead', 'High Potential', 'Cold Lead']
df_clean['Lead_Status'] = np.select(cond_status, label_status, default='Cold Lead')

# 3. ANALISIS CHURN RISK & SALES PRIORITY (Logika perbaikan High Potential)
cond_churn = [
    (df_clean['Days_Since_Last_Purchase'] > 90) & (df_clean['Days_Since_Last_Purchase'] != 999) & (df_clean['Email_Opened_flag'] == 0) & (df_clean['WhatsApp_Clicked_flag'] == 0),
    (df_clean['Days_Since_Last_Purchase'] > 60) & (df_clean['Days_Since_Last_Purchase'] != 999),
    (df_clean['Days_Since_Last_Purchase'] <= 60) | (df_clean['Purchase_Frequency_6M'] > 0)
]
label_churn = ['High', 'Medium', 'Low']
df_clean['Churn_Risk'] = np.select(cond_churn, label_churn, default='Low')

cond_priority = [
    (df_clean['Lead_Status'] == 'Hot Lead'),
    (df_clean['Lead_Status'] == 'Warm Lead') | (df_clean['Lead_Status'] == 'High Potential'),
    (df_clean['Lead_Status'] == 'Cold Lead') & (df_clean['Churn_Risk'] != 'High'),
    (df_clean['Churn_Risk'] == 'High')
]
label_priority = [
    'Hot Lead : follow up maksimal 1x24 jam',
    'Warm Lead : kirim edukasi, testimoni, dan promo',
    'Cold Lead : masukkan ke nurturing campaign',
    'Churn Risk : kirim promo reactivation'
]
df_clean['Sales_Priority'] = np.select(cond_priority, label_priority, default='Cold Lead : masukkan ke nurturing campaign')

# 4. KLASIFIKASI CUSTOMER STAGE & STRATEGI CAMPAIGN
cond_stage = [
    (df_clean['Purchase_Frequency_6M'] > 1),
    (df_clean['Purchase_Frequency_6M'] == 1),
    (df_clean['Add_To_Cart_flag'] == 1) | (df_clean['WhatsApp_Clicked_flag'] == 1) | (df_clean['Chat_Replied_flag'] == 1),
    (df_clean['View_Count'] >= top_view) | (df_clean['Pages_Visited'] >= top_pages) | (df_clean['Email_Opened_flag'] == 1),
    (df_clean['View_Count'] > 1) & (df_clean['View_Count'] < top_view),
    (df_clean['View_Count'] <= 1)
]
label_stage = ['Retention', 'Purchase', 'Intent', 'Consideration', 'Interest', 'Awareness']
df_clean['Customer_Stage'] = np.select(cond_stage, label_stage, default='Awareness')

cond_campaign = [
    (df_clean['Add_To_Cart_flag'] == 1) & (df_clean['Purchase_Frequency_6M'] == 0),
    (df_clean['Days_Since_Last_Purchase'] > 120) & (df_clean['Days_Since_Last_Purchase'] != 999),
    (df_clean['Customer_Stage'] == 'Consideration'),
    (df_clean['WhatsApp_Clicked_flag'] == 1) | (df_clean['Chat_Replied_flag'] == 1),
    (df_clean['Customer_Stage'] == 'Retention'),
    (df_clean['Total_Transaction_Value'] >= top_value)
]
label_campaign = [
    'WhatsApp reminder / Email follow up',
    'Promo reactivation (promo code)',
    'Tawarkan produk lainnya yang serupa atau bundling produk',
    'Sales Follow up',
    'Tawarkan loyalty program',
    'Berikan penawaran eksklusif'
]
df_clean['Campaign'] = np.select(cond_campaign, label_campaign, default='Testimoni customer')

# 5. ATUR REKOMENDASI PRODUK
def set_product_rec(row):
    item_view = str(row['Product_Viewed']).strip().lower()
    item_buy = str(row['Last_Product_Purchased']).strip().lower()

    if 'serum' in item_view or 'moisturizer' in item_buy:
        return 'serum, sunscreen, bundling skincare'
    elif 'moisturizer' in item_view:
        return 'moisturizer, serum, sunscreen'
    elif 'lipstick' in item_view or 'cushion' in item_view:
        return 'lipstick, cushion, bundling cosmetic'
    else:
        return 'serum, sunscreen, bundling skincare'

df_clean['Product_Recommendation'] = df_clean.apply(set_product_rec, axis=1)

# Drop flag pembantu, susun urutan kolom final A sampai G ke sisi paling kanan
df_clean = df_clean.drop(columns=[col + '_flag' for col in check_cols])

original_structure = list(df.columns)
new_target_structure = [
    'Lead_Score',
    'Lead_Status',
    'Sales_Priority',
    'Customer_Stage',
    'Churn_Risk',
    'Campaign',
    'Product_Recommendation'
]

final_mapping = original_structure + new_target_structure
df_clean = df_clean.reindex(columns=final_mapping)

# Save hasil data final
output_filename = 'CRM_Data_Final.xlsx'
df_clean.to_excel(output_filename, index=False)
print("Skrip sukses dijalankan. Data siap di Download.")